# 06 – Dự báo (Chuỗi thời gian)

Sử dụng file `timeseries_monthly.csv` để xây dựng dự báo doanh thu hàng tháng.  Mã chính nằm trong `src/models/forecasting.py` và kịch bản `scripts/run_forecasting.py`.  Chúng ta sẽ khảo sát xu hướng và tính mùa, đánh giá mô hình naive, ARIMA và Prophet, rồi vẽ biểu đồ dự báo tương lai.

*Lưu ý: hãy chắc chắn đã chạy `scripts/run_pipeline.py` để tạo file chuỗi thời gian.*

In [ ]:
# các thư viện chuẩn và cấu hình đường dẫn
import os
import sys

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import pandas as pd
from src.utils.config import load_config
from src.models import forecasting
from src.evaluation import metrics

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
cfg = load_config(os.path.join(ROOT, "configs", "params.yaml"))

# dùng thư mục processed (tạo bởi run_pipeline)
processed_dir = os.path.join(ROOT, cfg["paths"]["processed_dir"])

data_path = os.path.join(processed_dir, "timeseries_monthly.csv")
df = pd.read_csv(data_path)

# xem trước và thiết lập chỉ mục
df.head()

In [ ]:
date_col = cfg.get("forecasting", {}).get("date_col", "date")
value_col = cfg.get("forecasting", {}).get("value_col", "revenue")

df[date_col] = pd.to_datetime(df[date_col])
df = df.set_index(date_col).sort_index()
ts = df[value_col].asfreq("MS")

ts.plot(figsize=(10,4), title="Doanh thu hàng tháng")
plt.show()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

result = seasonal_decompose(ts.dropna(), model="additive")
result.plot()
plt.tight_layout()
plt.show()


In [ ]:
# chia train/test theo thời gian
h = cfg.get("forecasting", {}).get("test_periods", 6)
train = ts.iloc[:-h]
test = ts.iloc[-h:]

# naive
aive_pred = forecasting.naive_forecast(train, h)
# arima
arima_model = forecasting.train_arima(train, order=tuple(cfg.get("forecasting", {}).get("arima_order", (1,1,1))))
arima_pred = forecasting.forecast_arima(arima_model, h)

# so sánh
from pandas import concat
comparison = concat([test.rename("thực_tế"), naive_pred.rename("naive"), arima_pred.rename("arima")], axis=1)
comparison.plot(figsize=(8,4))
plt.title("Dự báo giai đoạn thử nghiệm")
plt.show()

# tính chỉ số
metrics_df = pd.DataFrame([
    {"model": "naive", **metrics.forecast_metrics(test, naive_pred)},
    {"model": "arima", **metrics.forecast_metrics(test, arima_pred)},
])
metrics_df.set_index("model")